# Web Search & Content Extraction Tools — Demo

Quick demos of `google_search` and `fetch_page_as_markdown`.

In [1]:
import sys, json
sys.path.insert(0, '..')

from tools.web_search import GoogleSearchInput, GoogleSearchResult, google_search
from tools.web_content import FetchPageInput, PageMarkdownResult, fetch_page_as_markdown
from IPython.display import Markdown, display

## 1. Fetch example.com — Simplest possible test

In [2]:
args = FetchPageInput(url="https://example.com")
raw = fetch_page_as_markdown(args)
result = PageMarkdownResult.model_validate_json(raw)

print(f"Title: {result.title}")
print(f"Error: {result.error}")
print(f"Markdown length: {len(result.markdown)} chars")
print("---")
display(Markdown(result.markdown))

[2026-03-19 17:05:38] INFO: Fetched (200) <GET https://example.com/> (referer: https://www.google.com/)


Title: Example Domain
Error: None
Markdown length: 168 chars
---


# Example Domain

This domain is for use in documentation examples without needing permission. Avoid use in operations.

[Learn more](https://iana.org/domains/example)


## 2. Google Search — "Pikachu pokemon"

In [3]:
args = GoogleSearchInput(query="Pikachu pokemon", max_results=5)
raw = google_search(args)
result = GoogleSearchResult.model_validate_json(raw)

print(f"Query: {result.query}")
print(f"Error: {result.error}")
print(f"Results: {result.total_returned}\n")

for i, r in enumerate(result.results):
    print(f"[{i+1}] {r.title}")
    print(f"    {r.url}")
    if r.snippet:
        print(f"    {r.snippet[:120]}")
    print()

[2026-03-19 17:05:44] INFO: Fetched (200) <GET https://www.google.com/search?q=Pikachu+pokemon&num=5&hl=en&sei=1x68aeGKC-iF9u8PrOXeqAo> (referer: https://www.google.com/)


Query: Pikachu pokemon
Error: None
Results: 5

[1] Pikachu (Pokémon) - Bulbapedia
    https://bulbapedia.bulbagarden.net/wiki/Pikachu_(Pok%C3%A9mon)
    Bulbapedia https://bulbapedia.bulbagarden.net › wiki › Pikachu_(Po...

[2] Pikachu – PokéWiki
    https://www.pokewiki.de/Pikachu
    pokewiki.de https://www.pokewiki.de › Pikachu

[3] Pikachu Pokédex: stats, moves, evolution & locations
    https://pokemondb.net/pokedex/pikachu
    Pokemon Database https://pokemondb.net › pokedex › pikachu

[4] 025 — Pikachu im Pokédex
    https://www.bisafans.de/pokedex/025.php
    Bisafans.de https://www.bisafans.de › pokedex

[5] Pikachu | Pokédex
    https://www.pokemon.com/us/pokedex/pikachu
    Pokemon.com https://www.pokemon.com › pokedex › pikachu



## 3. Google Search — Site-restricted to Bulbapedia

In [4]:
args = GoogleSearchInput(
    query="Charizard",
    site_restrict="bulbapedia.bulbagarden.net",
    max_results=5,
)
raw = google_search(args)
result = GoogleSearchResult.model_validate_json(raw)

print(f"Query: {result.query}")
print(f"Results: {result.total_returned}\n")

for i, r in enumerate(result.results):
    print(f"[{i+1}] {r.title}")
    print(f"    {r.url}\n")

[2026-03-19 17:06:33] INFO: Fetched (200) <GET https://www.google.com/search?q=site%3Abulbapedia.bulbagarden.net+Charizard&num=5&hl=en&sei=Bx-8aeaBOfyVxc8Px4Pd6Q4> (referer: https://www.google.com/)


Query: site:bulbapedia.bulbagarden.net Charizard
Results: 4

[1] Charizard (Pokémon) - Bulbapedia
    https://bulbapedia.bulbagarden.net/wiki/Charizard_(Pok%C3%A9mon)

[2] Charizard (TCG) - Bulbapedia
    https://bulbapedia.bulbagarden.net/wiki/Charizard_(TCG)

[3] Ash's Charizard - Bulbapedia, the community-driven Pokémon ...
    https://bulbapedia.bulbagarden.net/wiki/Ash%27s_Charizard

[4] Charizard (Pokémon)/Generation III learnset - Bulbapedia
    https://bulbapedia.bulbagarden.net/wiki/Charizard_(Pok%C3%A9mon)/Generation_III_learnset



## 4. Fetch Bulbapedia — Pikachu page (stealth mode)

Bulbapedia is behind Cloudflare, so we use `use_stealth=True`.

In [ ]:
args = FetchPageInput(
    url="https://bulbapedia.bulbagarden.net/wiki/Pikachu_(Pok%C3%A9mon)",
    css_selector="#mw-content-text",
    use_stealth=True,
)
raw = fetch_page_as_markdown(args)
result = PageMarkdownResult.model_validate_json(raw)

print(f"Title: {result.title}")
print(f"Error: {result.error}")
print(f"Markdown length: {len(result.markdown):,} chars")
print("---")
# Show first 2000 chars as rendered markdown
display(Markdown(result.markdown[:20000] + "\n\n*... (truncated) ...*"))

## 5. Fetch Wikipedia — Clean content extraction

In [7]:
args = FetchPageInput(
    url="https://en.wikipedia.org/wiki/Pok%C3%A9mon",
    css_selector="#bodyContent",
)
raw = fetch_page_as_markdown(args)
result = PageMarkdownResult.model_validate_json(raw)

print(f"Title: {result.title}")
print(f"Error: {result.error}")
print(f"Markdown length: {len(result.markdown):,} chars")
print("---")
display(Markdown(result.markdown[:3000] + "\n\n*... (truncated) ...*"))

[2026-03-19 17:08:25] INFO: Fetched (200) <GET https://en.wikipedia.org/wiki/Pok%C3%A9mon> (referer: https://www.google.com/)


Title: Pokémon - Wikipedia
Error: None
Markdown length: 302,460 chars
---


[![Page semi-protected](//upload.wikimedia.org/wikipedia/en/thumb/1/1b/Semi-protection-shackle.svg/20px-Semi-protection-shackle.svg.png)](/wiki/Wikipedia:Protection_policy#semi "This article is semi-protected.")

From Wikipedia, the free encyclopedia

Japanese media franchise

This article is about the media franchise as a whole. For the video game series, see [*Pokémon* (video game series)](/wiki/Pok%C3%A9mon_(video_game_series) "Pokémon (video game series)"). For the animated series, see [*Pokémon* (TV series)](/wiki/Pok%C3%A9mon_(TV_series) "Pokémon (TV series)"). For a list of creatures known as "Pokémon", see [List of Pokémon](/wiki/List_of_Pok%C3%A9mon "List of Pokémon"). For other uses, see [Pokémon (disambiguation)](/wiki/Pok%C3%A9mon_(disambiguation) "Pokémon (disambiguation)").

| Pokémon | |
| --- | --- |
| [![](//upload.wikimedia.org/wikipedia/commons/thumb/9/98/International_Pok%C3%A9mon_logo.svg/330px-International_Pok%C3%A9mon_logo.svg.png)](/wiki/File:International_Pok%C3%A9mon_logo.svg)   International franchise logo | |
| Created by | [Satoshi Tajiri](/wiki/Satoshi_Tajiri "Satoshi Tajiri") |
| Original work | [*Pocket Monsters Red* and *Pocket Monsters Green*](/wiki/Pok%C3%A9mon_Red,_Blue,_and_Yellow "Pokémon Red, Blue, and Yellow") (1996) |
| Owners | <br>[Nintendo](/wiki/Nintendo "Nintendo")[Creatures](/wiki/Creatures_(company) "Creatures (company)")[Game Freak](/wiki/Game_Freak "Game Freak")[[1]](#cite_note-pokemon.com-1) |
| Years | 1996–present |
| Print publications | |
| Comics | See [list of *Pokémon* manga](/wiki/List_of_Pok%C3%A9mon_manga "List of Pokémon manga") |
| Films and television | |
| Film(s) | See [list of *Pokémon* films](/wiki/List_of_Pok%C3%A9mon_films "List of Pokémon films") |
| Animated series | [*Pokémon*](/wiki/Pok%C3%A9mon_(TV_series) "Pokémon (TV series)") (1997–present) |
| Games | |
| Traditional | *[Pokémon Trading Card Game](/wiki/Pok%C3%A9mon_Trading_Card_Game "Pokémon Trading Card Game")* |
| Video game(s) | [*Pokémon* video game series](/wiki/Pok%C3%A9mon_(video_game_series) "Pokémon (video game series)") |
| Official website | |
| [Official hub](https://www.portal-pokemon.com/) | |


***Pokémon***[[a]](#cite_note-2)[[b]](#cite_note-3) is a Japanese [media franchise](/wiki/Media_franchise "Media franchise") consisting of [video games](/wiki/List_of_Pok%C3%A9mon_video_games "List of Pokémon video games"), [animated series](/wiki/Pok%C3%A9mon_(TV_series) "Pokémon (TV series)") and [films](/wiki/List_of_Pok%C3%A9mon_films "List of Pokémon films"), [a trading card game](/wiki/Pok%C3%A9mon_Trading_Card_Game "Pokémon Trading Card Game"), and other related media. The franchise takes place in a [shared universe](/wiki/Shared_universe "Shared universe") in which humans co-exist with creatures known as [Pokémon](/wiki/List_of_Pok%C3%A9mon "List of Pokémon"), a large variety of species endowed with special powers. The franchise's primary [target audience](/wiki/Target_audience "Target audience") is chil

*... (truncated) ...*

## 6. Search → Fetch pipeline

Search for something, then fetch the first result and show its markdown.

In [2]:
# Step 1: Search
search_args = GoogleSearchInput(
    query="Bulbasaur",
    site_restrict="bulbapedia.bulbagarden.net",
    max_results=1,
)
search_raw = google_search(search_args)
search_result = GoogleSearchResult.model_validate_json(search_raw)

if search_result.results:
    first = search_result.results[0]
    print(f"Top result: {first.title}")
    print(f"URL: {first.url}\n")

    # Step 2: Fetch that page
    fetch_args = FetchPageInput(
        url=first.url,
        css_selector="#mw-content-text",
        use_stealth=True,
    )
    fetch_raw = fetch_page_as_markdown(fetch_args)
    page = PageMarkdownResult.model_validate_json(fetch_raw)

    print(f"Page title: {page.title}")
    print(f"Content length: {len(page.markdown):,} chars")
    print("---")
    display(Markdown(page.markdown[:5000] + "\n\n*... (truncated) ...*"))
else:
    print("No results found.")

[2026-03-19 23:06:37] INFO: Fetched (200) <GET https://www.google.com/search?q=site%3Abulbapedia.bulbagarden.net+Bulbasaur&num=1&hl=en&sei=bHO8af7rNJmKxc8PiIqDgAU> (referer: https://www.google.com/)


Top result: Bulbasaur (Pokémon) - Bulbapedia
URL: https://bulbapedia.bulbagarden.net/wiki/Bulbasaur_(Pok%C3%A9mon)



[2026-03-19 23:06:40] INFO: Fetched (200) <GET https://bulbapedia.bulbagarden.net/wiki/Bulbasaur_(Pok%C3%A9mon)> (referer: https://www.google.com/)


Page title: Bulbasaur (Pokémon) - Bulbapedia, the community-driven Pokémon encyclopedia
Content length: 194,394 chars
---


---
title: "Bulbasaur (Pokémon) - Bulbapedia, the community-driven Pokémon encyclopedia"
url: "https://bulbapedia.bulbagarden.net/wiki/Bulbasaur_(Pok%C3%A9mon)"
timestamp: "2026-03-19T23:06:37.800363+01:00"
---



- For Pokémon GO information on this species, see [the game's section](#Pok%C3%A9mon_GO).
- | [Pokémon](/wiki/List_of_Pok%C3%A9mon_by_National_Pok%C3%A9dex_number "List of Pokémon by National Pokédex number") |
| --- | - [#0002: Ivysaur](/wiki/Ivysaur_(Pok%C3%A9mon) "Ivysaur (Pokémon)") [/wiki/Ivysaur_(Pok%C3%A9mon)](/wiki/Ivysaur_(Pok%C3%A9mon) "Ivysaur (Pokémon)") [→](/wiki/Ivysaur_(Pok%C3%A9mon) "Ivysaur (Pokémon)")
- This article is about the species. For a specific instance of this species, see [Bulbasaur (disambiguation)](/wiki/Bulbasaur_(disambiguation) "Bulbasaur (disambiguation)").

| | | **Bulbasaur**  
[Seed Pokémon](/wiki/Pok%C3%A9mon_category "Pokémon category") | **フシギダネ**  
*Fushigidane* |
| --- | --- | | [#0001](/wiki/List_of_Pok%C3%A9mon_by_National_Pok%C3%A9dex_number "List of Pokémon by National Pokédex number") |
| --- | --- |
| - [Bulbasaur](/wiki/File:0001Bulbasaur.png "Bulbasaur")
- [Images on the Bulbagarden Archives](https://archives.bulbagarden.net/wiki/Category:Bulbasaur "a:Category:Bulbasaur") | | | |
| --- | --- |
| **[Type](/wiki/Type "Type")**

- | [**Grass**](/wiki/Grass_(type) "Grass (type)") | [**Poison**](/wiki/Poison_(type) "Poison (type)") |
| --- | --- | | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") |
| --- | --- | | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") |
| --- | --- | | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") |
| --- | --- | | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") |
| --- | --- | | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") |
| --- | --- |
- | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") | [**Unknown**](/wiki/Unknown_(type) "Unknown (type)") |
| --- | --- | | |
| **[Abilities](/wiki/Ability "Ability")**

- [Overgrow](/wiki/Overgrow_(Ability) "Overgrow (Ability)") [Cacophony](/wiki/Cacophony_(Ability) "Cacophony (Ability)") [Cacophony](/wiki/Cacophony_(Ability) "Cacophony (Ability)") [Chlorophyll](/wiki/Chlorophyll_(Ability) "Chlorophyll (Ability)")
 Hidden Ability [Cacophony](/wiki/Cacophony_(Ability) "Cacophony (Ability)")
 Hidden Ability [Cacophony](/wiki/Cacophony_(Ability) "Cacophony (Ability)") [Cacophony](/wiki/Cacophony_(Ability) "Cacophony (Ability)") | |
| **[Gender ratio](/wiki/List_of_Pok%C3%A9mon_by_gender_ratio "List of Pokémon by gender ratio")**

- Unknown
- [87.5% male, 12.5% female](/wiki/Category:Pok%C3%A9mon_with_a_gender_ratio_of_seven_males_to_one_female "Category:Pokémon with a gender ratio of seven males to one female") | **[Catch rate](/wiki/Catch_rate "Catch rate")**

| 45 (11.9%) |
| --- | |
| **[Breeding](/wiki/Pok%C3%A9mon_breeding "Pokémon breeding")**

- **[Egg Groups](/wiki/Egg_Group "Egg Group")**

| [Monster](/wiki/Monster_(Egg_Group) "Monster (Egg Group)") and [Grass](/wiki/Grass_(Egg_Group) "Grass (Egg Group)") |
| --- | **[Hatch time](/wiki/Egg_cycle "Egg cycle")**

| 20 cycles |
| --- | | |
| **[Height](/wiki/List_of_Pok%C3%A9mon_by_height "List of Pokémon by height")**

- 2'04" 0.7 m
- Bulbasaur
- 0'0" 0 m
- {{{form2}}}
- 0'0" 0 m
- {{{form3}}}
- 0'0" 0 m
- {{{form4}}}
- 0'0" 0 m
- {{{form5}}}
- 0'0" 0 m
- {{{form6}}}
- 0'0" 0 m
- {{{form7}}} | **[Weight](/wiki/Weight "Weight")**

- 15.2 lbs. 6.9 kg
- Bulbasaur
- 0 lbs. 0 kg
- {{{form2}}}
- 0 lbs. 0 kg
- {{{form3}}}
- 0 lbs. 0 kg
- {{{form4}}}
- 0 lbs. 0 kg
- {{{form5}}}
- 0 lbs. 0 kg
- {{{form6}}}
- 0 lbs. 0 kg
- {{{form7}}} |
| **[Mega Stone](/wiki/Mega_Stone "Mega Stone")**

| [[\|]] | [[\|]] |
| --- | --- | | |
| **[Base experience yield](/wiki/Experience "Experience")**

| 64  
Gen. I-IV | Unknown  
IV | 64  
V+ |
| --- | --- | --- | | **[Leveling rate](/wiki/Experience "Experience")**

| Medium Slow |
| --- | |
| **[EV yield](/wiki/List_of_Pok%C3%A9mon_by_effort_value_yield "List of Pokémon by effort value yield")**

- Total: 1
- Bulbasaur
- 0
HP 0
Atk 0
Def 1
Sp.Atk 0
Sp.Def 0
Speed
- 0
HP 0
Atk 0
Def 0
Sp.Atk 0
Sp.Def 0
Speed
- 0
HP 0
Atk 0
Def 0
Sp.Atk 0
Sp.Def 0
Speed
- 0
HP 0
Atk 0
Def 0
Sp.Atk 0
Sp.Def 0
Speed | |
| **[Shape](/wiki/List_of_Pok%C3%A9mon_by_shape "List of Pokémon by shape")**

| [![](https://archives.bulbagarden.net/media/upload/thumb/c/cc/Body08.png/32px-Body08.png)](/wiki/File:Body08.png) |
| --- | | **[Footprint](/wiki/Footprint "Footprint")**

| [![](https://archives.bulbagarden.net/media/upload/d/d1/F0001.png)](/wiki/File:F0001.png) | [![](https://archives.bulbagarden.net/media/upload/e/e3/None.png)](/wiki/File:None.png)  
{{{form2}}} |
| --- | --- | |
| **[Pokédex color](/wiki/List_of_Pok%C3%A9mon_by

*... (truncated) ...*

## 7. Vector Database Ingestion

Ingesting a webpage into the vector database using Chonkie for chunking and ChromaDB for storage.

In [2]:
from tools.web_vector_db import ingest_web_page, IngestWebPageArgs

args = IngestWebPageArgs(
    url="https://bulbapedia.bulbagarden.net/wiki/Charmander_(Pok%C3%A9mon)",
    css_selector="#mw-content-text",
    use_stealth=True
)
result = ingest_web_page(args)
print(result)

[2026-03-19 23:15:37] INFO: Fetched (200) <GET https://bulbapedia.bulbagarden.net/wiki/Charmander_(Pok%C3%A9mon)> (referer: https://www.google.com/)


Ingested 407 chunks from 'Charmander (Pokémon) - Bulbapedia, the community-driven Pokémon encyclopedia'. Available via `query_web_content`.


## 8. Vector Database Querying

Retrieving semantically relevant chunks from the ingested web corpus.

In [3]:
from tools.web_vector_db import query_web_content, QueryWebContentArgs

query_args = QueryWebContentArgs(
    query="What is the flame on Charmander's tail?",
    n_results=10
)
query_result = query_web_content(query_args)
display(Markdown(query_result))

### Charmander (Pokémon) - Bulbapedia, the community-driven Pokémon encyclopedia
**Source:** https://bulbapedia.bulbagarden.net/wiki/Charmander_(Pok%C3%A9mon)
**Chunk:** 22/407

 #229 The flame on its tail indicates Charmander's life force. If it is healthy, the flame burns brightly. *(Pokémon Blue, Gold, or Yellow inserted)* 

---

### Charmander (Pokémon) - Bulbapedia, the community-driven Pokémon encyclopedia
**Source:** https://bulbapedia.bulbagarden.net/wiki/Charmander_(Pok%C3%A9mon)
**Chunk:** 18/407

Charmander protecting the flame on its tail from the rain

Charmander is a bipedal, [reptilian](https://en.wikipedia.org/wiki/reptile "wp:reptile") [Pokémon](/wiki/Pok%C3%A9mon_(species) "Pokémon (species)") with a primarily orange body and blue eyes. Its underside from the chest down and the soles of its feet are cream-colored. It has two small fangs visible in its upper jaw and two smaller fangs in its lower jaw. A fire burns at the tip of this Pokémon's slender tail, which has blazed there since Charmander's birth.

The flame can indicate Charmander's health and mood, burning brightly when the Pokémon is strong, weakly when it is exhausted, wavering when it is happy, and blazing when it is enraged. It is said that Charmander would die if its flame were to go out. Howe

---

### Charmander (Pokémon) - Bulbapedia, the community-driven Pokémon encyclopedia
**Source:** https://bulbapedia.bulbagarden.net/wiki/Charmander_(Pok%C3%A9mon)
**Chunk:** 19/407

The flame can indicate Charmander's health and mood, burning brightly when the Pokémon is strong, weakly when it is exhausted, wavering when it is happy, and blazing when it is enraged. It is said that Charmander would die if its flame were to go out. However, if the Pokémon is healthy, the flame will continue to burn even if it gets a bit wet and is said to steam in the rain. Charmander can be found in hot, [mountainous areas](/wiki/List_of_Pok%C3%A9mon_by_habitat#Mountain_Pok%C3%A9mon "List of Pokémon by habitat"). It has been recently seen living in the [Terarium](/wiki/Terarium "Terarium") of [Blueberry Academy](/wiki/Blueberry_Academy "Blueberry Academy"). However, it is found far more often in the ownership of [Trainers](/wiki/Pok%C3%A9mon_Trainer "Pokémon Trainer"). As seen in [Pokémon Snap](/wiki/Pok%C3%A9mon_Snap "Pokémon Snap") and [New Pokémon Snap](/wiki/New_Pok%C3%A9mon_Snap "New Pokémon Snap"), Charmander exhibits pack behavior, calling others of its species if it finds food and watching the flames on each other's tails to ensure they don't go out. As shown in [Pokémon Sleep](/wiki/Pok%C3%A9mon_Sleep "Pokémon Sleep"), Charmander is known to sleep while curled up. Supposedly, it draws warmth from the flame on its tail. During quiet nights, the sounds of its flame can be heard by listening carefully while Charmander sleeps.[[1]](#cite_note-1)  


### Evolution

Charmander [evolves](/wiki/Evolution "Evolution") into [Charmeleon](/wiki/Charmeleon_(Pok%C3%A9mon) "Charmeleon (Pokémon)"), which evolves into [Charizard](/wiki/Charizard_(Pok%C3%A9mon) "Charizard (Pokémon)").

(For specifics on this Pokémon's Evolution i

---

### Charmander (Pokémon) - Bulbapedia, the community-driven Pokémon encyclopedia
**Source:** https://bulbapedia.bulbagarden.net/wiki/Charmander_(Pok%C3%A9mon)
**Chunk:** 23/407

The flame on its tail indicates Charmander's life force. If it is healthy, the flame burns brightly. *(Pokémon Blue, Gold, or Yellow inserted)*  #— |  | [Kanto](/wiki/List_of_Pok%C3%A9mon_by_Kanto_Pok%C3%A9dex_number "List of Pokémon by Kanto Pokédex number")  
 #004 

---

### Charmander (Pokémon) - Bulbapedia, the community-driven Pokémon encyclopedia
**Source:** https://bulbapedia.bulbagarden.net/wiki/Charmander_(Pok%C3%A9mon)
**Chunk:** 346/407

A Charmander appeared in a flashback in *[Sword and Shield... The Legends Awaken! (Part 1)](/wiki/JNM13 "JNM13")*, under the ownership of [Leon](/wiki/Leon "Leon"). It is now his Charizard in the present day.

## In the TCG

*Main article: [Charmander (TCG)](/wiki/Charmander_(TCG) "Charmander (TCG)")*

## Other appearances

### [Super Smash Bros. Brawl](/wiki/Super_Smash_Bros._Brawl "Super Smash Bros. Brawl")

Charmander appears as a trophy.

#### Trophy information

*"A Lizard Pokémon. It just downright likes hot stuff. The always-burning tail indicates its mood--waving gently when content and burning intensely when angry. If the tail were to go out, it would be the end of Charmander's life. Its tail is believed to emit steam when it rains. Charmander evolves into Charmeleon by leveling up."*

### [Super Smash Bros. Ultimate](/wiki/Super_Smash_Bros._Ultimate "Super Smash Bros. Ultimate")

Charmander appears as a [Spirit](https://www.ssbwiki.com/Spirits_(characters) "sbw:Spirits (characters)").

### [*POKÉMON Detective Pikachu*](/wiki/POK%C3%89MON_Detective_Pikachu "POKÉMON Detective Pikachu")

Multiple Charmander appeared in [*POKÉMON Detective Pikachu*](/wiki/POK%C3%89MON_Detective_Pikachu "POKÉMON Detective Pikachu").

### Celestial

Charmander in the music video for [Celestial](/wiki/Celestial "Celestial")

Charmander appeared in the music video for [Celestial](/wiki/Celestial "Celestial").

## Trivia



---

### Charmander (Pokémon) - Bulbapedia, the community-driven Pokémon encyclopedia
**Source:** https://bulbapedia.bulbagarden.net/wiki/Charmander_(Pok%C3%A9mon)
**Chunk:** 223/407

[Connected](/wiki/Connection_Orb "Connection Orb") to: [Bulbasaur](/wiki/Bulbasaur_(Pok%C3%A9mon) "Bulbasaur (Pokémon)"), [Squirtle](/wiki/Squirtle_(Pok%C3%A9mon) "Squirtle (Pokémon)"), [Chimchar](/wiki/Chimchar_(Pok%C3%A9mon) "Chimchar (Pokémon)")
[Connection Orb](/wiki/Connection_Orb "Connection Orb") Set: 1
- **Phrases**
- Normal The fire at the tip of my tail shows my energy!
- Low HP (< 50%) Argh… It's getting tough…
- **Phrases if the hero**
- Normal (All right! I'll give it my all!)
- Low HP (< 50%) (This is hard…but I need to hang in there!)
- **Phrases if the partner**
- Normal Let's give it our all!
- Low HP (< 50%) I'm OK… I can take it a little more… | | | | | |
| - **[Pokémon Ranger](/wiki/Pok%C3%A9mon_Ranger_(video_game) "Pokémon Ranger (video game)")**

- Group: (Burn ×1)
- Loops: 3 Min. exp.: 25 Max. exp.: 39
- **Browser entry [R-140](/wiki/List_of_Pok%C3%A9mon_by_Fiore_Browser_number "List of Pokémon by Fiore Browser number")**
- *Charmander shudders while breathing fire. It is the very picture of vitality.* | |

---

### Charmander (Pokémon) - Bulbapedia, the community-driven Pokémon encyclopedia
**Source:** https://bulbapedia.bulbagarden.net/wiki/Charmander_(Pok%C3%A9mon)
**Chunk:** 17/407

  * [4.3 The Electric Tale of Pikachu](#The_Electric_Tale_of_Pikachu)
  * [4.4 Magical Pokémon Journey](#Magical_Pok%C3%A9mon_Journey)
  * [4.5 Pokémon Zensho](#Pok%C3%A9mon_Zensho)
  * [4.6 Pokémon: Yeah! I Got Pokémon!](#Pok%C3%A9mon:_Yeah!_I_Got_Pok%C3%A9mon!)
  * [4.7 Pocket Monsters XY: The Legend of the Pokémon Dragon King](#Pocket_Monsters_XY:_The_Legend_of_the_Pok%C3%A9mon_Dragon_King)
  * [4.8 Movie adaptations](#Movie_adaptations)
  * [4.9 Pokémon Journeys](#Pok%C3%A9mon_Journeys)
- [5 In the TCG](#In_the_TCG)
- [6 Other appearances](#Other_appearances)
  * [6.1 Super Smash Bros. Brawl](#Super_Smash_Bros._Brawl)
    + [6.1.1 Trophy information](#Trophy_information)
  * [6.2 Super Smash Bros. Ultimate](#Super_Smash_Bros._Ultimate)
  * [6.3 *POKÉMON Detective Pikachu*](#POK%C3%89MON_Detective_Pikachu)
  * [6.4 Celestial](#Celestial)
- [7 Trivia](#Trivia)
  * [7.1 Concept and development](#Concept_and_development)
  * [7.2 Design variations](#Design_variations)
  * [7.3 Origin](#Origin)
    + [7.3.1 Name origin](#Name_origin)
- [8 In other languages](#In_other_languages)
- [9 See also](#See_also)
- [10 References](#References)
- [11 External links](#External_links)

## Biology

Charmander protecting the flame on its tail from the rain

Charmander is a bipedal, [reptilian](https://en.wikipedia.org/wiki/reptile "wp:reptile") [Pokémon](/wiki/Pok%C3%A9mon_(species) "Pokémon (species)") with a primarily orange body and blue eyes. Its 

---

### Charmander (Pokémon) - Bulbapedia, the community-driven Pokémon encyclopedia
**Source:** https://bulbapedia.bulbagarden.net/wiki/Charmander_(Pok%C3%A9mon)
**Chunk:** 224/407

(Burn ×1)
- Loops: 3 Min. exp.: 25 Max. exp.: 39
- **Browser entry [R-140](/wiki/List_of_Pok%C3%A9mon_by_Fiore_Browser_number "List of Pokémon by Fiore Browser number")**
- *Charmander shudders while breathing fire. It is the very picture of vitality.* | | | | | |
| - **[Pokémon Ranger: Shadows of Almia](/wiki/Pok%C3%A9mon_Ranger:_Shadows_of_Almia "Pokémon Ranger: Shadows of Almia")**

- Group: (Burn ×1)
- **Browser entry [R-137](/wiki/List_of_Pok%C3%A9mon_by_Almia_Browser_number "List of Pokémon by Almia Browser number")**
- *It attacks by spitting embers.* | | | | | |
| - **[Pokémon Ranger: Guardian Signs](/wiki/Pok%C3%A9mon_Ranger:_Guardian_Si

---

### Charmander (Pokémon) - Bulbapedia, the community-driven Pokémon encyclopedia
**Source:** https://bulbapedia.bulbagarden.net/wiki/Charmander_(Pok%C3%A9mon)
**Chunk:** 329/407

 | | | |


 For other sprites and images, please see [Charmander images on the Bulbagarden Archives](https://archives.bulbagarden.net/wiki/Category:Charmander "a:Category:Charmander").




## In animation

### Main series



---

### Charmander (Pokémon) - Bulbapedia, the community-driven Pokémon encyclopedia
**Source:** https://bulbapedia.bulbagarden.net/wiki/Charmander_(Pok%C3%A9mon)
**Chunk:** 13/407

  
{{{form2}}} 
- On Smogon Pokédex:
<br> [Generation I](https://www.smogon.com/dex/rb/pokemon/charmander/) [Generation II](https://www.smogon.com/dex/gs/pokemon/charmander/) [Generation III](https://www.smogon.com/dex/rs/pokemon/charmander/) [Generation IV](https://www.

---